In [0]:
#Transform raw login events into customer-level usage metrics. 
# Import Libraries 
from pyspark.sql.functions import *

In [0]:
# Load Bronze Usage Table 
usage = spark.table("customer_usage_500_k")

In [0]:
# Inspect the Data 
display(usage)

EventID,CustomerID,LoginDate,SessionMinutes
E0,171006,2025-11-25,65
E1,149238,2025-04-22,16
E2,158393,2025-08-01,172
E3,135509,2025-07-22,105
E4,135801,2025-04-22,40
E5,176792,2024-03-21,82
E6,158905,2024-08-26,146
E7,111404,2025-05-04,138
E8,189103,2025-09-29,136
E9,125993,2025-02-21,129


In [0]:
# Print schema 

usage.printSchema()

root
 |-- EventID: string (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- LoginDate: date (nullable = true)
 |-- SessionMinutes: long (nullable = true)



In [0]:
# Count Records 
print("Total Usage Records:", usage.count())


Total Usage Records: 500000


In [0]:
# Check Missing Values 
usage.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in usage.columns
]).show()

+-------+----------+---------+--------------+
|EventID|CustomerID|LoginDate|SessionMinutes|
+-------+----------+---------+--------------+
|      0|         0|        0|             0|
+-------+----------+---------+--------------+



In [0]:
# Remove Duplicate Events (Each login event should be unique.) 
silver_usage = usage.dropDuplicates(["EventID"])

In [0]:
print("Original:", usage.count())

print("After Removing Duplicates:", silver_usage.count())

Original: 500000
After Removing Duplicates: 500000


In [0]:
# Handle Missing Values

# Replace missing session time with 0.  
silver_usage = silver_usage.fillna({
    "SessionMinutes":0
})


In [0]:
##  Convert Login Date 
# Current datatype 
silver_usage = silver_usage.withColumn(
    "LoginDate",
    to_date(col("LoginDate"))
)

In [0]:
# Verify 
silver_usage.printSchema()

root
 |-- EventID: string (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- LoginDate: date (nullable = true)
 |-- SessionMinutes: long (nullable = false)



In [0]:
# Validate Session Minutes

# Negative session duration is impossible. Also remove unrealistic sessions. Assume Maximum 12 hours duration  

silver_usage = silver_usage.filter(
    (col("SessionMinutes") >= 0) &
    (col("SessionMinutes") <= 720)
)

In [0]:
# Create Login Month Useful for monthly reporting. 
silver_usage = silver_usage.withColumn(
    "LoginMonth",
    date_format(col("LoginDate"),"yyyy-MM")
)

In [0]:
# Aggregate Customer Usage

# Now convert event-level records into customer-level metrics. 
silver_usage = silver_usage.groupBy("CustomerID").agg(
    count("EventID").alias("TotalLogins"),
    sum("SessionMinutes").alias("LifetimeMinutes"),
    avg("SessionMinutes").alias("AvgSession"),
    max("LoginDate").alias("LastLoginDate"),
    min("LoginDate").alias("FirstLoginDate")
)

In [0]:
# Calculate Active Days , Customer lifetime in days. 
silver_usage = silver_usage.withColumn(
    "ActiveDays",
    datediff(
        col("LastLoginDate"),
        col("FirstLoginDate")
    )
)

In [0]:
# Create Login Frequency , Average logins per active day. 
silver_usage = silver_usage.withColumn(
    "LoginFrequency",
    when(
        col("ActiveDays")==0,
        col("TotalLogins")
    ).otherwise(
        round(
            col("TotalLogins") /
            col("ActiveDays"),
            2
        )
    )
)

In [0]:
# Customer Activity Level , Create engagement categories. 
silver_usage = silver_usage.withColumn(
    "UsageLevel",
    when(col("TotalLogins") >= 200,"HIGH")
    .when(col("TotalLogins") >= 100,"MEDIUM")
    .otherwise("LOW")
)

In [0]:
# Average Session Category 
silver_usage = silver_usage.withColumn(
    "SessionCategory",
    when(col("AvgSession") >= 60,"LONG")
    .when(col("AvgSession") >= 30,"MEDIUM")
    .otherwise("SHORT")
)

In [0]:
# Recent Login Flag , Has the customer logged in during the last 30 days? 

silver_usage = silver_usage.withColumn(
    "RecentlyActive",
    when(
        datediff(current_date(),col("LastLoginDate")) <=30,
        1
    ).otherwise(0)
)

In [0]:
# Verify Output 
display(silver_usage)

CustomerID,TotalLogins,LifetimeMinutes,AvgSession,LastLoginDate,FirstLoginDate,ActiveDays,LoginFrequency,UsageLevel,SessionCategory,RecentlyActive
171006,7,690,98.57142857142857,2025-11-25,2024-01-06,689,0.01,LOW,LONG,0
149238,6,393,65.5,2025-11-27,2024-02-04,662,0.01,LOW,LONG,0
158393,6,613,102.16666666666667,2025-08-12,2024-03-24,506,0.01,LOW,LONG,0
135509,8,898,112.25,2025-12-24,2024-01-06,718,0.01,LOW,LONG,0
135801,4,374,93.5,2025-12-05,2024-02-14,660,0.01,LOW,LONG,0
176792,6,469,78.16666666666667,2025-11-04,2024-01-01,673,0.01,LOW,LONG,0
158905,1,146,146.0,2024-08-26,2024-08-26,0,1.0,LOW,LONG,0
111404,7,510,72.85714285714286,2025-11-10,2024-06-18,510,0.01,LOW,LONG,0
189103,4,441,110.25,2025-09-29,2025-07-07,84,0.05,LOW,LONG,0
125993,8,674,84.25,2025-08-02,2024-02-25,524,0.02,LOW,LONG,0


In [0]:
# Verify Schema 
silver_usage.printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- TotalLogins: long (nullable = false)
 |-- LifetimeMinutes: long (nullable = true)
 |-- AvgSession: double (nullable = true)
 |-- LastLoginDate: date (nullable = true)
 |-- FirstLoginDate: date (nullable = true)
 |-- ActiveDays: integer (nullable = true)
 |-- LoginFrequency: double (nullable = true)
 |-- UsageLevel: string (nullable = false)
 |-- SessionCategory: string (nullable = false)
 |-- RecentlyActive: integer (nullable = false)



In [0]:
# Save Silver Usage 
silver_usage.write.mode("overwrite").saveAsTable("silver_usage")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6369503641629204>, line 2
      1 # Save Silver Usage 
----> 2 silver_usage.write.mode("overwrite").saveAsTable("silver_usage")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id = self._user_id
   1537 self._set_command_in_

In [0]:
spark.sql("DROP TABLE IF EXISTS silver_usage")

DataFrame[]

In [0]:
silver_usage.write.mode("overwrite").saveAsTable("silver_usage")

In [0]:
spark.sql("DESCRIBE DETAIL silver_usage").show(truncate=False)

+------+------------------------------------+------------------------------+-----------+--------+-----------------------+-------------------+----------------+-----------------+--------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                          |description|location|createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                                                                                                                                                                 |minReaderVersion|minWriterVersion|tableFeatures                            |statistics                     

In [0]:
# Validate Saved Table 
display(spark.table("silver_usage"))

CustomerID,TotalLogins,LifetimeMinutes,AvgSession,LastLoginDate,FirstLoginDate,ActiveDays,LoginFrequency,UsageLevel,SessionCategory,RecentlyActive
143787,4,241,60.25,2025-11-21,2024-07-11,498,0.01,LOW,LONG,0
139020,4,422,105.5,2025-11-28,2024-01-09,689,0.01,LOW,LONG,0
186574,4,287,71.75,2025-04-19,2024-01-16,459,0.01,LOW,LONG,0
168935,6,584,97.33333333333333,2025-12-28,2024-01-31,697,0.01,LOW,LONG,0
180741,8,541,67.625,2025-10-20,2024-02-09,619,0.01,LOW,LONG,0
199671,5,327,65.4,2025-06-24,2024-06-29,360,0.01,LOW,LONG,0
143102,5,320,64.0,2025-12-05,2024-07-03,520,0.01,LOW,LONG,0
177809,11,1051,95.54545454545455,2025-11-26,2024-01-07,689,0.02,LOW,LONG,0
159531,4,295,73.75,2025-10-12,2025-03-20,206,0.02,LOW,LONG,0
170389,5,361,72.2,2025-12-12,2024-03-01,651,0.01,LOW,LONG,0
